# 0.0. Setup

In [1]:
import os
from pathlib import Path

import re
import pickle
import importlib
import numpy as np
import pandas as pd
import openpyxl as opxl
import matplotlib.pyplot as plt
from typing import Optional, Dict, Any, List

from difflib import SequenceMatcher
from sklearn.impute import SimpleImputer, KNNImputer

# None means 'no limit'
pd.set_option('display.max_columns', None)

PROJECT_DIR = Path.cwd().parent
if 'modules' not in os.listdir(Path.cwd()):
    os.chdir(PROJECT_DIR)

DATA_DIR = 'data'
DATA_PUBLIC_DIR = 'data/public'
DATA_PRIVATE_DIR = 'data/private'
DATA_PROCESSED_DIR = 'data/processed'
OUTPUT_DIR = 'output'

## 0.1. Modules

In [2]:
# We use the .py modules from the AAAI submission
from modules import public_furnitures_processor as slp
from modules import private_furniture_processor as pfp

# 1.0. Load data

## 1.1. Public seats

In [3]:
# Initialize processor
processor = slp.PublicFurnituresProcessor()

# Process data to long format
pub_long_data = processor.process()

# Get summary
summary = processor.get_summary()

INFO:modules.public_furnitures_processor:Starting seat-learner ratio data processing
INFO:modules.public_furnitures_processor:Loading data from data/public/SY 2023-2024 SEAT-LEARNER RATIO.xlsx
INFO:modules.public_furnitures_processor:Trimmed whitespaces from 13 string columns
INFO:modules.public_furnitures_processor:Loaded 47821 records with 26 columns
INFO:modules.public_furnitures_processor:Transforming seat data from wide to long format
INFO:modules.public_furnitures_processor:Trimmed whitespaces from 2 string columns
INFO:modules.public_furnitures_processor:Created long format data with 54360 records
INFO:modules.public_furnitures_processor:Education levels: ['Elementary', 'Junior High School', 'Senior High School']
INFO:modules.public_furnitures_processor:Processing completed successfully
/workspace/project_paaral/modules/public_furnitures_processor.py:221: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass ob

In [4]:
print(pub_long_data.shape)
display(pub_long_data.head())

(54360, 3)


,school_id,education_level,seat_count
0,100001,Elementary,195
1,100002,Elementary,731
2,100003,Elementary,192
3,100004,Elementary,134
4,100005,Elementary,44


In [5]:
fname = 'processed_public_seats.parquet'
outpath = os.path.join(OUTPUT_DIR, fname)
pub_long_data.to_parquet(outpath)

## 1.2. Private seats

In [6]:
# Initialize processor
processor = pfp.PrivateFurnitureProcessor(verbose=True)

# Process data to long format
priv_long_data = processor.process()

INFO:modules.private_furniture_processor:Starting private furniture data processing
INFO:modules.private_furniture_processor:Loading data from data/private/priv_classroom_furniture.xlsx
INFO:modules.private_furniture_processor:Trimmed whitespaces from 6 string columns
INFO:modules.private_furniture_processor:Loaded 11579 records with 24 columns
INFO:modules.private_furniture_processor:Transforming furniture data from wide to long format
INFO:modules.private_furniture_processor:Trimmed whitespaces from 3 string columns
/workspace/project_paaral/modules/private_furniture_processor.py:268: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  .replace(
INFO:modules.private_furniture_processor:Created long format data with 43586 records
INFO:modules.private_furniture_processor:Grade lev

In [7]:
print(priv_long_data.shape)
display(priv_long_data.head())

(43586, 6)


,school_id,raw_grade_level,furniture_type,furniture_count,alt_furniture_counts,grade_level
8,400001,JHS,Desks,4,8,Junior High School
10,400001,JHS,Arm Chairs,36,36,Junior High School
12,400001,SHS,Desks,2,4,Senior High School
14,400001,SHS,Arm Chairs,19,19,Senior High School
26,400002,JHS,Arm Chairs,250,250,Junior High School


In [8]:
priv_long_data.dtypes

school_id               string[python]
raw_grade_level               category
furniture_type                  object
furniture_count                  int64
alt_furniture_counts             int64
grade_level                   category
dtype: object

In [9]:
fname = 'processed_private_seats.parquet'
outpath = os.path.join(OUTPUT_DIR, fname)
priv_long_data.to_parquet(outpath)